mounted to drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
!mkdir -p /content/drive/MyDrive/llm_finance

In [ ]:

!pip install -q transformers datasets peft accelerate bitsandbytes safetensors sentencepiece evaluate rouge_score nltk


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00


loading and preparing the data set from hugging face library

In [ ]:
# Create prompt/response CSVs and copy them to Drive
from datasets import load_dataset
import pandas as pd, re, os, shutil
from sklearn.model_selection import train_test_split

OUT_PREP = "finance_data/prepared"
DRIVE_PREP = "/content/drive/MyDrive/llm_finance/prepared"
os.makedirs(OUT_PREP, exist_ok=True)
os.makedirs(DRIVE_PREP, exist_ok=True)

print("Loading dataset from Hugging Face...")
ds = load_dataset("FinLang/investopedia-instruction-tuning-dataset")
print("Splits:", list(ds.keys()))

def clean(s):
    if s is None:
        return ""
    s = str(s).replace("\n"," ").replace("\r"," ").strip()
    s = re.sub(r"\s+"," ", s)
    return s

rows = []
print("Preparing prompt/response rows (this may take 30-60s)...")
for split in ds.keys():
    for ex in ds[split]:
        title = clean(ex.get("Title",""))
        context = clean(ex.get("Context",""))
        question = clean(ex.get("Question",""))
        answer = clean(ex.get("Answer",""))
        qa = clean(ex.get("Question-Answer",""))
        # extract question from combined QA if needed
        if not question and qa and "Question:" in qa and "Answer:" in qa:
            try:
                question = qa.split("Question:",1)[1].split("Answer:",1)[0].strip()
            except:
                question = question
        if not question:
            continue
        response = answer if answer else (qa.split("Answer:",1)[1].strip() if "Answer:" in qa else qa)
        if len(response) < 3:
            continue
        parts = []
        if title:
            parts.append("Title: " + re.sub(r'<.*?>','', title))
        if context:
            parts.append("Context: " + context)
        parts.append("Question: " + question)
        prompt = "Instruction: Answer the following finance question based on the provided context (if any).\n\n" + "\n\n".join(parts) + "\n\nResponse:"
        rows.append({"prompt": prompt, "response": response})

df = pd.DataFrame(rows)
print("Total prepared examples:", len(df))

# split 80/10/10
train, test = train_test_split(df, test_size=0.10, random_state=42, shuffle=True)
train, val = train_test_split(train, test_size=0.1111, random_state=42, shuffle=True)  # ~80/10/10
print("Sizes -> train:", len(train), "val:", len(val), "test:", len(test))

# save locally
train.to_csv(os.path.join(OUT_PREP, "train.csv"), index=False)
val.to_csv(os.path.join(OUT_PREP, "val.csv"), index=False)
test.to_csv(os.path.join(OUT_PREP, "test.csv"), index=False)
print("Saved CSVs to", OUT_PREP)

# copy to Drive for persistence
try:
    shutil.rmtree(DRIVE_PREP, ignore_errors=True)
    shutil.copytree(OUT_PREP, DRIVE_PREP)
    print("Copied CSVs to Drive at", DRIVE_PREP)
except Exception as e:
    print("Warning: could not copy to Drive:", e)

# show 3 samples for sanity-check
print("\nThree sample prompts/responses (truncated):")
for i, row in train.sample(3, random_state=42).iterrows():
    print("-"*60)
    print("PROMPT (first 300 chars):\n", row["prompt"][:300].replace("\n"," "))
    print("RESPONSE (first 300 chars):\n", row["response"][:300])


Loading dataset from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/277M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/206461 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22940 [00:00<?, ? examples/s]

Splits: ['train', 'test']
Preparing prompt/response rows (this may take 30-60s)...
Total prepared examples: 229103
Sizes -> train: 183284 val: 22908 test: 22911
Saved CSVs to finance_data/prepared
Copied CSVs to Drive at /content/drive/MyDrive/llm_finance/prepared

Three sample prompts/responses (truncated):
------------------------------------------------------------
PROMPT (first 300 chars):
 Instruction: Answer the following finance question based on the provided context (if any).  Title: Primary Offering: What it is, How it Works  Context: It is a means for a private company to raise equity capital through financial markets to expand its business operations. A primary offering can also
RESPONSE (first 300 chars):
 A primary offering is a means for a private company to raise equity capital through financial markets to expand its business operations. Companies often use it to finance their expansion, especially growing and mature private companies.
-----------------------------------

In [ ]:
# STEP 2: Tokenize prompt+response and save HF tokenized dataset (sample train -> 10k)
!pip install -q transformers datasets sentencepiece tqdm

import os, random, shutil
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from tqdm import tqdm

# CONFIG - edit only if you want different sample size / model / length
PREPARED_DIR = "finance_data/prepared"
OUT_TOKENIZED_DIR = "finance_data/tokenized"
DRIVE_TOKENIZED_DIR = "/content/drive/MyDrive/llm_finance/tokenized"
MODEL_NAME = "gpt2"
MAX_LENGTH = 256
SAMPLE_TRAIN_N = 10000   # set None to use full train
BATCH_SIZE = 64
SEED = 42

os.makedirs(OUT_TOKENIZED_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DRIVE_TOKENIZED_DIR), exist_ok=True)
random.seed(SEED)

print("Loading CSVs...")
train_df = pd.read_csv(os.path.join(PREPARED_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(PREPARED_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(PREPARED_DIR, "test.csv"))
print("Original train size:", len(train_df), "val:", len(val_df), "test:", len(test_df))

# Sample train if requested
if SAMPLE_TRAIN_N and SAMPLE_TRAIN_N < len(train_df):
    train_df = train_df.sample(n=SAMPLE_TRAIN_N, random_state=SEED).reset_index(drop=True)
    print("Sampled train ->", len(train_df))
else:
    print("Using full train ->", len(train_df))

# Init tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_prompt_response(prompt, response, max_length=MAX_LENGTH):
    # tokenize prompt and response separately (no special tokens)
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    r_ids = tokenizer(response, add_special_tokens=False)["input_ids"]

    # truncate prompt if too long (keep tail), reserve rest for response
    if len(p_ids) >= max_length:
        p_ids = p_ids[-(max_length//2):]
    avail = max_length - len(p_ids)
    if avail <= 0:
        r_ids = []
    else:
        if len(r_ids) > avail:
            r_ids = r_ids[:avail]

    input_ids = p_ids + r_ids
    attention_mask = [1] * len(input_ids)

    # pad to max_length
    pad_len = max_length - len(input_ids)
    if pad_len > 0:
        input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
        attention_mask = attention_mask + [0] * pad_len

    # labels: -100 for prompt tokens and padding, actual ids for response tokens
    labels = [-100] * len(p_ids) + r_ids + [-100] * pad_len
    assert len(input_ids) == max_length and len(labels) == max_length
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

def df_to_tokenized_ds(df, name):
    ds = Dataset.from_pandas(df.reset_index(drop=True))
    def map_fn(batch):
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for prompt, response in zip(batch["prompt"], batch["response"]):
            tok = tokenize_prompt_response(str(prompt), str(response), max_length=MAX_LENGTH)
            out["input_ids"].append(tok["input_ids"])
            out["attention_mask"].append(tok["attention_mask"])
            out["labels"].append(tok["labels"])
        return out
    ds_tok = ds.map(map_fn, batched=True, batch_size=BATCH_SIZE, remove_columns=ds.column_names, desc=f"Tokenizing {name}")
    ds_tok = ds_tok.with_format("torch")
    print(f"Tokenized {name}: {len(ds_tok)} examples")
    return ds_tok

# Tokenize (may take a few minutes for 10k)
train_tok = df_to_tokenized_ds(train_df, "train")
val_tok   = df_to_tokenized_ds(val_df, "validation")
test_tok  = df_to_tokenized_ds(test_df, "test")

hf_tok = DatasetDict({"train": train_tok, "validation": val_tok, "test": test_tok})
hf_tok.save_to_disk(OUT_TOKENIZED_DIR)
print("Saved tokenized dataset to", OUT_TOKENIZED_DIR)

# Copy tokenized dataset to Drive for persistence
try:
    shutil.rmtree(DRIVE_TOKENIZED_DIR, ignore_errors=True)
    shutil.copytree(OUT_TOKENIZED_DIR, DRIVE_TOKENIZED_DIR)
    print("Copied tokenized dataset to Drive at", DRIVE_TOKENIZED_DIR)
except Exception as e:
    print("Warning: could not copy tokenized dataset to Drive:", e)

# Print one example summary
ex = train_tok[0]
decoded = tokenizer.decode([i for i,m in zip(ex["input_ids"], ex["attention_mask"]) if m==1], skip_special_tokens=True)
print("\nExample tokenized preview (train[0]):")
print("input_ids length:", len(ex["input_ids"]))
print("non-pad tokens:", sum(ex["attention_mask"]))
print("decoded (non-pad preview):", decoded[:400].replace("\n"," "))
print("labels sample (first 60):", ex["labels"][:60])


Loading CSVs...
Original train size: 183284 val: 22908 test: 22911
Sampled train -> 10000


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Tokenized test: 22911 examples


Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/22908 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/22911 [00:00<?, ? examples/s]

Saved tokenized dataset to finance_data/tokenized
Copied tokenized dataset to Drive at /content/drive/MyDrive/llm_finance/tokenized

Example tokenized preview (train[0]):
input_ids length: 256
non-pad tokens: tensor(256)
decoded (non-pad preview): Instruction: Answer the following finance question based on the provided context (if any).  Title: Primary Offering: What it is, How it Works  Context: It is a means for a private company to raise equity capital through financial markets to expand its business operations. A primary offering can also include debt issuance. Primary offerings are usually a way for a growing company to raise financing
labels sample (first 60): tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
     

In [ ]:
# STEP 3: Train LoRA adapters (4-bit if possible; fallback to fp16)
# Paste & run this full cell in Colab.

!pip install -q bitsandbytes peft transformers accelerate safetensors

import os, torch, shutil, warnings
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from transformers import default_data_collator
from peft import LoraConfig, get_peft_model, TaskType

warnings.filterwarnings("ignore")



In [ ]:
TOKENIZED_DIR = "finance_data/tokenized"                 # tokenized dataset
MODEL_NAME = "gpt2"                                      # base model (used for tokenizer)
OUT_DIR = "./finance_lora"                               # local save
DRIVE_OUT = "/content/drive/MyDrive/llm_finance/finance_lora_adapter"
USE_4BIT = True                                          # try 4-bit (recommended)
NUM_EPOCHS = 3                                           # set higher to train longer
PER_DEVICE_BATCH = 1
GRADIENT_ACCUMULATION = 16
LEARNING_RATE = 2e-4
FP16 = True                                              # use mixed precision
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
print("Device CUDA available:", torch.cuda.is_available())
print("Loading tokenized dataset from:", TOKENIZED_DIR)
hf_ds = load_from_disk(TOKENIZED_DIR)
print({k: len(v) for k,v in hf_ds.items()})

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Device CUDA available: True
Loading tokenized dataset from: finance_data/tokenized
{'train': 10000, 'validation': 22908, 'test': 22911}


In [ ]:
# Try loading model quantized in 4-bit using BitsAndBytesConfig (preferred)
model = None
if USE_4BIT:
    try:
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4"
        )
        print("Attempting to load model with 4-bit quantization (BitsAndBytesConfig)...")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_cfg,
            device_map="auto",
            trust_remote_code=False,
            low_cpu_mem_usage=True
        )
        print("Loaded model in 4-bit.")
    except Exception as e:
        print("4-bit load failed — falling back to fp16. Error:", e)
        model = None

if model is None:
    print("Loading model in fp16 (non-quantized fallback)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True
    )
    print("Loaded model in fp16.")


Attempting to load model with 4-bit quantization (BitsAndBytesConfig)...
Loaded model in 4-bit.


In [ ]:
# Apply LoRA (PEFT)
print("Applying LoRA adapters (r={}, alpha={})...".format(LORA_R, LORA_ALPHA))
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["c_attn","c_proj"],   # GPT-2 style modules
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Applying LoRA adapters (r=8, alpha=32)...
trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475


In [ ]:
# TrainingArguments (minimal set for widest compatibility)
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    fp16=FP16,
    logging_steps=200,
    save_total_limit=3,
    remove_unused_columns=False,
    report_to="none",
)


In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_ds["train"],
    eval_dataset=hf_ds["validation"],
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

print("Starting training (this can take minutes per epoch). Output dir:", OUT_DIR)
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting training (this can take minutes per epoch). Output dir: ./finance_lora


Step,Training Loss
200,1.406000
400,1.275100
600,1.232500
800,1.196000
1000,1.176800
1200,1.163300
1400,1.148200
1600,1.150700
1800,1.131900


TrainOutput(global_step=1875, training_loss=1.204400634765625, metrics={'train_runtime': 2795.9679, 'train_samples_per_second': 10.73, 'train_steps_per_second': 0.671, 'total_flos': 3956751728640000.0, 'train_loss': 1.204400634765625, 'epoch': 3.0})

In [ ]:

# Save adapter locally and to Drive (adapter files are small)
adapter_path_local = os.path.join(OUT_DIR, "adapter")
model.save_pretrained(adapter_path_local)
print("Saved adapter locally to:", adapter_path_local)

# copy to Drive
try:
    shutil.rmtree(DRIVE_OUT, ignore_errors=True)
    shutil.copytree(adapter_path_local, DRIVE_OUT)
    print("Copied adapter to Drive at:", DRIVE_OUT)
except Exception as e:
    print("Warning: could not copy adapter to Drive:", e)

print("Training complete.")


Saved adapter locally to: ./finance_lora/adapter
Copied adapter to Drive at: /content/drive/MyDrive/llm_finance/finance_lora_adapter
Training complete.


In [ ]:
# Smoke test: 5 example generations (very fast)
import torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_NAME = "gpt2"
ADAPTER_DIR = "./finance_lora/adapter"        # local adapter (you saved here)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# load base + adapter
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

def gen(prompt, max_new_tokens=120, temp=0.0):
    inputs = tokenizer(prompt, return_tensors="pt").to(next(model.parameters()).device)
    with torch.no_grad():
        out = model.generate(input_ids=inputs["input_ids"], max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id, do_sample=(temp>0.0), temperature=temp)
    gen = tokenizer.decode(out[0].tolist()[inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    return gen

val_df = pd.read_csv("finance_data/prepared/val.csv")
sample = val_df.sample(n=5, random_state=42).reset_index(drop=True)
for i in range(len(sample)):
    p = sample.loc[i, "prompt"]
    g = sample.loc[i, "response"]
    print("="*80)
    print("PROMPT (truncated):\n", p[:400].replace("\n"," "))
    print("\nGOLD:\n", g[:400])
    print("\nMODEL (greedy):\n", gen(p, max_new_tokens=160, temp=0.0)[:800])
    print("\nMODEL (creative):\n", gen(p, max_new_tokens=160, temp=0.7)[:800])


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


PROMPT (truncated):
 Instruction: Answer the following finance question based on the provided context (if any).  Title: Evolutionary Economics: Overview, History, Examples  Context: The term was first coined by Thorstein Veblen (1857-1929), an American economist and sociologist. Traditional economic theories generally view people and governmental institutions as entirely rational actors. Evolutionary economics differs

GOLD:
 Thorstein Veblen (1857-1929)

MODEL (greedy):
 Thorstein Veblen (1857-1929), an American economist and sociologist. Traditional economic theories generally view people and governmental institutions as entirely rational actors. Evolutionary economics differs from rational choice theory in that it focuses on complex psychological factors as key drivers of the economy. Evolutionary economists believe the economy is dynamic, constantly changing, and chaotic, rather than always tending toward a state of equilibrium. The creation of goods and the procurement of supplies

In [ ]:
# Quick eval (100 samples): ROUGE + token-overlap F1
!pip install -q evaluate rouge_score nltk
import torch, pandas as pd, nltk
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import evaluate
nltk.download('punkt', quiet=True)

MODEL_NAME = "gpt2"
ADAPTER_DIR = "./finance_lora/adapter"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

def generate_cont(prompt, max_new_tokens=120, temp=0.0):
    inputs = tokenizer(prompt, return_tensors="pt").to(next(model.parameters()).device)
    with torch.no_grad():
        out = model.generate(input_ids=inputs["input_ids"], max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id, do_sample=(temp>0.0), temperature=temp)
    gen = tokenizer.decode(out[0].tolist()[inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    return gen

val_df = pd.read_csv("finance_data/prepared/val.csv")
N = min(100, len(val_df))
sample_df = val_df.sample(n=N, random_state=42).reset_index(drop=True)

rouge = evaluate.load("rouge")
preds, refs = [], []
for i, row in sample_df.iterrows():
    prompt = row["prompt"]
    gold = str(row["response"])
    gen = generate_cont(prompt, max_new_tokens=120, temp=0.0)
    preds.append(gen); refs.append(gold)
print("Generated", len(preds), "samples.")

rouge_res = rouge.compute(predictions=preds, references=refs)
print("\nROUGE:", rouge_res)

# token F1
import nltk
def token_f1(a,b):
    a_tokens = nltk.word_tokenize(a.lower())
    b_tokens = nltk.word_tokenize(b.lower())
    if len(a_tokens)==0 or len(b_tokens)==0: return 0.0
    common = set(a_tokens) & set(b_tokens)
    if not common: return 0.0
    prec = len(common)/len(a_tokens); rec = len(common)/len(b_tokens)
    return 2*prec*rec/(prec+rec)
f1s = [token_f1(p,r) for p,r in zip(preds, refs)]
print("Token-overlap F1 (avg):", sum(f1s)/len(f1s))


Generated 100 samples.

ROUGE: {'rouge1': np.float64(0.3269716849603931), 'rouge2': np.float64(0.22664437800925294), 'rougeL': np.float64(0.2849344182584237), 'rougeLsum': np.float64(0.2842612883521901)}


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')   # <-- this fixes the LookupError you saw
print("NLTK tokenizer resources ready.")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


NLTK tokenizer resources ready.


In [ ]:
def token_f1(a,b):
    a_tokens = nltk.word_tokenize(a.lower())
    b_tokens = nltk.word_tokenize(b.lower())
    if len(a_tokens)==0 or len(b_tokens)==0:
        return 0.0
    common = set(a_tokens) & set(b_tokens)
    if not common: return 0.0
    prec = len(common)/len(a_tokens)
    rec = len(common)/len(b_tokens)
    return 2*prec*rec/(prec+rec)

f1s = [token_f1(p,r) for p,r in zip(preds, refs)]
print("Token-overlap F1 (avg):", sum(f1s)/len(f1s))


Token-overlap F1 (avg): 0.2571553950200208


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, default_data_collator
from peft import PeftModel
from datasets import load_from_disk
import torch, os

TOKENIZED_DIR = "finance_data/tokenized"
BASE_MODEL = "gpt2"
EXISTING_ADAPTER = "./finance_lora/adapter"
OUTPUT_DIR = "./finance_lora_continued"
NUM_EXTRA_EPOCHS = 5

hf_ds = load_from_disk(TOKENIZED_DIR)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, EXISTING_ADAPTER)
for n,p in model.named_parameters():
    p.requires_grad = "lora" in n.lower()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EXTRA_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=200,
    save_total_limit=3,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=hf_ds["train"],
                  eval_dataset=hf_ds["validation"],
                  tokenizer=tokenizer,
                  data_collator=default_data_collator)

trainer.train()
model.save_pretrained(os.path.join(OUTPUT_DIR,"adapter"))
print("✅ Continued adapter saved to", OUTPUT_DIR)


The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Step,Training Loss
200,1.115600
400,1.106300
600,1.098100
800,1.073100
1000,1.067400
1200,1.060500
1400,1.045200
1600,1.050600
1800,1.038200
2000,1.010600


✅ Continued adapter saved to ./finance_lora_continued
